# jiuzhang-sdk 云平台 GBS 任务与本地 GBS 示例

本 Notebook 演示 `jiuzhang-sdk` 的两类能力：

1. **云平台 GBS 任务**：在本地或 Jupyter 中通过 SDK 向九章云平台提交 GBS 实验，由平台侧执行并返回结果。
2. **本地 GBS 采样**：完全在用户本机运行，不经过九章云平台，结果由本地数值算法生成。

云平台任务默认不会自动提交。准备好工作台凭证后，将环境变量 `JIUZHANG_RUN_CLOUD_TASK=1` 即可执行完整云平台流程。

## 1. 工作台凭证与运行开关

运行云平台任务前，请先在九章云平台工作台获取：

- `api_key`：API Key，作为鉴权凭证。
- `project_id`：项目 ID，用于关联实验任务。
- `quantum_computer_id`：设备编码，例如 `PH_QC_04`。

推荐通过环境变量传入，避免把真实凭证写入 Notebook。

In [1]:
import os
from pprint import pprint

import jiuzhang

BASE_URL = os.getenv("JIUZHANG_BASE_URL", "https://cloud.jiuzhangqt.com/api/v1").strip()
API_KEY = os.getenv("JIUZHANG_API_KEY", "").strip()
PROJECT_ID = os.getenv("JIUZHANG_PROJECT_ID", "").strip()
QUANTUM_COMPUTER_ID = os.getenv("JIUZHANG_QUANTUM_COMPUTER_ID", "PH_QC_04").strip()

MT_VALUE = int(os.getenv("JIUZHANG_MT_VALUE", "500"))
PUMP_ENERGY_NJ = float(os.getenv("JIUZHANG_PUMP_ENERGY_NJ", "4.6"))
SQUEEZING_PARAM = float(os.getenv("JIUZHANG_SQUEEZING_PARAM", "0.35"))
TASK_NAME = os.getenv("JIUZHANG_TASK_NAME", "GBS experiment from sdk_usage.ipynb").strip()

POLL_INTERVAL = float(os.getenv("JIUZHANG_POLL_INTERVAL", "2.0"))
POLL_TIMEOUT = float(os.getenv("JIUZHANG_POLL_TIMEOUT", "300.0"))
RUN_CLOUD_TASK = os.getenv("JIUZHANG_RUN_CLOUD_TASK", "0").strip() == "1"


def mask_secret(value: str) -> str:
    if len(value) >= 10:
        return value[:7] + "***" + value[-4:]
    return "***" if value else ""


config_summary = {
    "sdk_version": jiuzhang.__version__,
    "base_url": BASE_URL,
    "api_key": mask_secret(API_KEY),
    "project_id": PROJECT_ID,
    "quantum_computer_id": QUANTUM_COMPUTER_ID,
    "mt_value": MT_VALUE,
    "pump_energy_nj": PUMP_ENERGY_NJ,
    "squeezing_param": SQUEEZING_PARAM,
    "run_cloud_task": RUN_CLOUD_TASK,
}
pprint(config_summary)

if RUN_CLOUD_TASK and not (API_KEY and PROJECT_ID and QUANTUM_COMPUTER_ID):
    raise RuntimeError(
        "Cloud task execution requires JIUZHANG_API_KEY, JIUZHANG_PROJECT_ID, "
        "and JIUZHANG_QUANTUM_COMPUTER_ID."
    )

{'api_key': '',
 'base_url': 'https://cloud.jiuzhangqt.com/api/v1',
 'mt_value': 500,
 'project_id': '',
 'pump_energy_nj': 4.6,
 'quantum_computer_id': 'PH_QC_04',
 'run_cloud_task': False,
 'sdk_version': '0.1.0',
 'squeezing_param': 0.35}


## 2. 初始化云平台客户端

In [2]:
from jiuzhang import CloudClient

# 未配置真实凭证时仍可初始化客户端并执行本地部分。
# 只有 RUN_CLOUD_TASK=True 时才会实际访问云平台；该模式会在第 1 节检查真实凭证。
client = CloudClient(
    base_url=BASE_URL,
    api_key=API_KEY or None,
    timeout=60.0,
)

print("CloudClient initialized:", client.base_url)

CloudClient initialized: https://cloud.jiuzhangqt.com/api/v1


## 3. 构造 GBS 实验参数

In [3]:
from jiuzhang import GBSParams

params = GBSParams(
    project_id=PROJECT_ID or "EXP-demo-project",
    quantum_computer_id=QUANTUM_COMPUTER_ID,
    mt=MT_VALUE,
    pump_energy_nj=PUMP_ENERGY_NJ,
    squeezing_param=SQUEEZING_PARAM,
    task_name=TASK_NAME,
)

print("Cloud payload:")
pprint(params.to_cloud_payload())
print("input_mode_count:", params.input_mode_count())
print("output_mode_count:", params.output_mode_count())

Cloud payload:
{'mt_value': 500,
 'project_id': 'EXP-demo-project',
 'pump_energy_nj': 4.6,
 'quantum_computer_id': 'PH_QC_04',
 'squeezing_param': 0.35,
 'task_name': 'GBS experiment from sdk_usage.ipynb'}
input_mode_count: 1500
output_mode_count: 5220


## 4. 复杂度预估

复杂度预估用于提交实验前查看平台侧预估信息。未开启 `JIUZHANG_RUN_CLOUD_TASK=1` 时，本单元只打印跳过说明。

In [4]:
estimate = None

if RUN_CLOUD_TASK:
    estimate = client.estimate_runtime(
        quantum_computer_id=params.quantum_computer_id,
        mt_value=params.mt,
        pump_energy_nj=params.pump_energy_nj,
    )
    print("Complexity estimate:")
    pprint(estimate)
else:
    print("Skipped cloud estimate. Set JIUZHANG_RUN_CLOUD_TASK=1 to run it.")

Skipped cloud estimate. Set JIUZHANG_RUN_CLOUD_TASK=1 to run it.


## 5. 提交实验

In [5]:
task = None
task_id = None

if RUN_CLOUD_TASK:
    task = client.submit_task(
        project_id=params.project_id,
        task_name=params.task_name,
        quantum_computer_id=params.quantum_computer_id,
        mt_value=params.mt,
        pump_energy_nj=params.pump_energy_nj,
        squeezing_param=params.squeezing_param,
    )
    task_id = task.get("data", {}).get("task_id") or task.get("data", {}).get("taskId")
    print("Task submitted. task_id:", task_id)
else:
    print("Skipped cloud submission. Set JIUZHANG_RUN_CLOUD_TASK=1 to submit a task.")

Skipped cloud submission. Set JIUZHANG_RUN_CLOUD_TASK=1 to submit a task.


## 6. 轮询并解析结果

In [6]:
import time

from jiuzhang import parse_gbs_result

raw_result = None
result = None


def extract_status(payload):
    data = payload.get("data", payload) if isinstance(payload, dict) else {}
    if not isinstance(data, dict):
        return None
    tianyan_response = data.get("tianyan_response")
    if isinstance(tianyan_response, dict) and tianyan_response.get("taskStatus") is not None:
        return str(tianyan_response.get("taskStatus"))
    for key in ("status", "taskStatus"):
        if data.get(key) is not None:
            return str(data.get(key))
    return None


TERMINAL_STATUSES = {"SUCCESS", "SUCCEEDED", "FAILED", "CANCELLED", "CANCELED", "2", "3"}

if RUN_CLOUD_TASK and task_id is not None:
    deadline = time.monotonic() + POLL_TIMEOUT
    while True:
        raw_result = client.get_result(task_id)
        status = extract_status(raw_result)
        print("Current status:", status)
        if status in TERMINAL_STATUSES:
            break
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Task polling timed out: {task_id}")
        time.sleep(POLL_INTERVAL)

    result = parse_gbs_result(raw_result)
    print("GBSResult status:", result.status_name)
    print("sample_count:", result.sample_count)
else:
    print("Skipped cloud polling because no cloud task was submitted.")

Skipped cloud polling because no cloud task was submitted.


## 7. 展示和绘制云平台结果

In [7]:
if result is None:
    print("No cloud result available. Run with JIUZHANG_RUN_CLOUD_TASK=1 to display cloud curves.")
else:
    from jiuzhang.jupyter import display_gbs_result

    display_gbs_result(result)

No cloud result available. Run with JIUZHANG_RUN_CLOUD_TASK=1 to display cloud curves.


In [8]:
if result is None or not result.result_map_points:
    print("No resultMapPoints available for plotting.")
else:
    import matplotlib.pyplot as plt

    labels = {
        "experimental": "experimental",
        "ground_truth": "ground truth",
        "squashed": "squashed",
        "thermal": "thermal",
        "distinguishable": "distinguishable",
    }

    plt.figure(figsize=(9, 5))
    for key, label in labels.items():
        points = result.result_map_points.get(key)
        if not points:
            continue
        xs = [point[0] for point in points]
        ys = [point[1] for point in points]
        plt.plot(xs, ys, label=label)

    plt.xlabel("Photon count")
    plt.ylabel("Probability")
    plt.title("Cloud GBS result distribution")
    plt.legend()
    plt.tight_layout()
    plt.show()

No resultMapPoints available for plotting.


## 8. 本地 GBS 采样

本地 GBS 采样完全在用户本机运行，不访问九章云平台，也不会产生云平台任务 ID。结果由 SDK 本地数值后端 在本机根据邻接矩阵、平均光子数、探测器类型、截断参数和随机种子计算生成。

In [9]:
import numpy as np

from jiuzhang.local.gbs import (
    random_adjacency_matrix,
    sample_gbs,
    samples_to_distribution,
)

graph = random_adjacency_matrix(8, scale=0.16, seed=7)
print("Adjacency matrix shape:", graph.shape)
print(np.round(graph, 4))

samples = sample_gbs(
    graph,
    shots=24,
    mean_photon_count=1.0,
    detector="pnr",
    cutoff=4,
    max_photons=12,
    seed=123,
)

print("Local samples preview:")
for row in samples[:10]:
    print(row)

distribution = samples_to_distribution(samples)
print("Top local patterns:")
for pattern, probability in sorted(distribution.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(pattern, f"{probability:.3f}")

Adjacency matrix shape: (8, 8)
[[0.     0.1436 0.1241 0.036  0.048  0.1398 0.0008 0.1314]
 [0.1436 0.     0.0485 0.0445 0.0408 0.0712 0.0807 0.0886]
 [0.1241 0.0485 0.     0.1582 0.0344 0.0256 0.098  0.007 ]
 [0.036  0.0445 0.1582 0.     0.1007 0.0823 0.0795 0.0396]
 [0.048  0.0408 0.0344 0.1007 0.     0.0006 0.1328 0.0247]
 [0.1398 0.0712 0.0256 0.0823 0.0006 0.     0.0146 0.0866]
 [0.0008 0.0807 0.098  0.0795 0.1328 0.0146 0.     0.024 ]
 [0.1314 0.0886 0.007  0.0396 0.0247 0.0866 0.024  0.    ]]


Local samples preview:
[0, 2, 0, 0, 0, 2, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 1, 0, 0, 0]
[0, 0, 1, 0, 1, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 1, 0, 0, 0, 0, 0]
Top local patterns:
(0, 0, 0, 0, 0, 0, 0, 0) 0.583
(1, 0, 1, 0, 0, 0, 0, 0) 0.083
(0, 2, 0, 0, 0, 2, 0, 0) 0.042
(0, 0, 0, 1, 1, 0, 0, 0) 0.042
(0, 0, 1, 0, 1, 0, 0, 0) 0.042
(2, 0, 1, 0, 0, 0, 0, 1) 0.042
(1, 0, 1, 2, 0, 0, 0, 0) 0.042
(0, 0, 1, 1, 0, 0, 0, 0) 0.042
(1, 0, 0, 0, 0, 0, 0, 1) 0.042
(0, 0, 0, 0, 1, 0, 1, 0) 0.042


## 9. 本地数学计算与 Program 序列化

In [ ]:
from jiuzhang.local.gbs import (
    GBSProgram,
    dumps_ir,
    hafnian,
    to_blackbird,
    to_xir,
    torontonian,
)

adjacency_2x2 = np.array([[0.0, 1.0], [1.0, 0.0]])
tor_matrix = np.array([[0.10, 0.02], [0.02, 0.10]])

print("Hafnian:", hafnian(adjacency_2x2))
print("Torontonian:", torontonian(tor_matrix))

program = (
    GBSProgram(modes=8)
    .squeezing([0.35, 0.32, 0.30, 0.28, 0.25, 0.22, 0.20, 0.18])
    .edge(0, 1, 0.15)
    .edge(1, 2, 0.10)
    .edge(2, 3, 0.12)
    .edge(4, 5, 0.11)
    .edge(5, 6, 0.09)
    .edge(6, 7, 0.13)
    .measure_fock(shots=240)
)

json_ir = dumps_ir(program)
blackbird_ir = to_blackbird(program)
xir_ir = to_xir(program)

print("JSON IR preview:")
print("\n".join(json_ir.splitlines()[:12]))
print("\n本地程序文本 preview:")
print("\n".join(blackbird_ir.splitlines()[:8]))
print("\nXIR preview:")
print("\n".join(xir_ir.splitlines()[:8]))

## 10. 释放资源

In [11]:
client.close()
print("CloudClient closed.")

CloudClient closed.
